In [1]:
# Uncomment line below to install exlib
# !pip install diskcache
import sys; 

ROOT_DIR = '../..'
sys.path.append(f'{ROOT_DIR}/src')


import openai
import os

# with open(f"{ROOT_DIR}/API_KEY.txt", "r") as file:
#     api_key = file.read().strip()
# with open(f"{ROOT_DIR}/API_KEY.txt", "r") as file:
#     api_key = file.read().strip()
import json
with open(f"{ROOT_DIR}/API_KEYS2.json", "r") as file:
    api_keys = json.load(file)

os.environ['OPENAI_API_KEY'] = api_keys['OPENAI_API_KEY']
os.environ['ANTHROPIC_API_KEY'] = api_keys['ANTHROPIC_API_KEY']
os.environ['GOOGLE_API_KEY'] = api_keys['GOOGLE_API_KEY']
os.environ['CACHE_DIR'] = os.path.join(ROOT_DIR, 'cache_dir3')

# MassMaps

In [2]:
import torch
from datasets import load_dataset

test_dataset = load_dataset("BrachioLab/massmaps-cosmogrid-100k", split='test')
test_dataset.set_format('torch', columns=['input', 'label'])

In [3]:
# import importlib
import sys; sys.path.append("../src")
# import massmaps
# importlib.reload(massmaps)
from massmaps import MassMapsExample
from massmaps import massmap_to_pil_norm, get_llm_generated_answer, get_llm_output
from massmaps import isolate_individual_features, distill_relevant_features, calculate_expert_alignment_scores
from llms import load_model

In [4]:
from tqdm.auto import tqdm
import json

In [5]:
# model = 'gpt-4o'
models = [
    'gpt-4o',
    # 'claude-3-5-sonnet-latest',
    # 'gemini-2.0-flash',
    # 'o1'
]

eval_model_name = 'gpt-4o'
eval_model = load_model(eval_model_name)



In [6]:
methods = [
    'vanilla', 
    # 'cot', 
    # 'socratic', 
    # 'subq'
]

In [7]:
import torch
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [8]:
from massmaps import isolate_individual_features_expert

In [9]:
import json
import copy
from tqdm.auto import tqdm

for model in models:
    print(f"=== Using model {model} ===")
    for method in methods:
        print(f"=== Using method {method} ===")

        load_path = os.path.join(ROOT_DIR, f'results/{method}/massmaps_{model}.json')
        save_path = os.path.join(ROOT_DIR, f'results/{method}/massmaps_{model}_{eval_model_name}.json')

        with open(load_path) as input_file:
            results = json.load(input_file)

        new_results = []

        num_examples = 3 #len(results)
        for di in tqdm(range(num_examples)):
            result = results[di]
            
            example_dict = result
            
            example = MassMapsExample(
                input = torch.tensor(example_dict['input']).to(device),
                answer = example_dict['answer'],
                llm_answer = example_dict['llm_answer'],
                llm_explanation = example_dict['llm_explanation'],
            )
            
            # isolate individual features
            expert_claims = isolate_individual_features_expert(
                example.llm_explanation, model=eval_model
            )
            
            if expert_claims is None:
                continue
            
            example.expert_claims = [claim.strip() for claim in expert_claims]
            
            claims = isolate_individual_features(
                example.llm_explanation, model=eval_model
            )
            
            if claims is None:
                continue
            
            example.claims = [claim.strip() for claim in claims]
            
            new_results.append(example)
            
#             # distill relevant features
#             relevant_claims = distill_relevant_features(
#                 example.input, 
#                 example.llm_answer,
#                 example.claims,
#                 model=eval_model
#             )
#             example.relevant_claims = relevant_claims

#             # calculate expert alignment scores
#             align_infos = calculate_expert_alignment_scores(
#                 example.relevant_claims, 
#                 eval_model,
#             )

#             alignable_claims = [info["Claim"] for info in align_infos]
#             alignment_categories = [info["Category"] for info in align_infos]
#             aligned_category_ids = [info["Category ID"] for info in align_infos]
#             alignment_scores = [info["Alignment"] for info in align_infos]
#             alignment_raws = [info["Alignment Raw"] for info in align_infos]
#             alignment_reasonings = [info["Reasoning"] for info in align_infos]
            
#             example.alignable_claims = alignable_claims
#             example.alignment_categories = alignment_categories
#             example.aligned_category_ids = aligned_category_ids
#             example.alignment_scores = alignment_scores
#             example.alignment_raws = alignment_raws
#             example.alignment_reasonings = alignment_reasonings
            
#             # save
#             save_dict = {}
#             for k, v in example.__dict__.items():
#                 save_dict[k] = v if not isinstance(v, torch.Tensor) else v.cpu().numpy().tolist()
#             # with open(save_path, 'wt') as output_file:
#             #     json.dump(save_dict, output_file)

#             new_results.append(save_dict)


#         with open(save_path, 'wt') as output_file:
#             json.dump(new_results, output_file, indent=4)

=== Using model gpt-4o ===
=== Using method vanilla ===


  0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
di = 0
print('===== CLAIMS =====')
for claim in new_results[di].claims:
    print(claim)

print('===== EXPERT CLAIMS =====')
for claim in new_results[di].expert_claims:
    print(claim)

===== CLAIMS =====
The weak lensing map shows a mixture of colors with a dominant presence of gray and red.
Gray regions indicate mass density fluctuations around zero.
Red regions indicate mass density fluctuations above zero.
Some yellow regions suggest areas with higher peaks of density exceeding 2.9 standard deviations.
The presence of gray, red, and some yellow structures indicates a relatively rich and varied mass distribution.
A relatively rich and varied mass distribution likely points to a moderate value of both Omega_m and sigma_8.
===== EXPERT CLAIMS =====
1. Lensing Peak (Cluster) Abundance: Some yellow regions suggest areas with higher peaks of density exceeding 2.9 standard deviations, which indicates a relatively rich and varied mass distribution.
2. Void Size and Frequency: N/A.
3. Filament Thickness and Sharpness: N/A.
4. Fine-Scale Clumpiness: The mixture of colors, with a dominant presence of gray and red and some yellow regions, indicates a relatively rich and varie

In [11]:
di = 1
print('===== CLAIMS =====')
for claim in new_results[di].claims:
    print(claim)

print('===== EXPERT CLAIMS =====')
for claim in new_results[di].expert_claims:
    print(claim)

===== CLAIMS =====
The weak lensing map shows a mix of underdense regions and overdense regions, indicated by blue and red and yellow colors, respectively.
The colors in the map indicate matter inhomogeneities.
The presence of significant red areas suggests moderate fluctuations.
The yellow spots in the map indicate highly overdense regions.
The yellow spots suggest a higher fluctuation amplitude.
The distribution and intensity of underdense and overdense regions imply moderate matter density and fluctuations.
===== EXPERT CLAIMS =====
1. Lensing Peak (Cluster) Abundance: The yellow spots indicate highly overdense regions, suggesting higher fluctuation amplitude.
2. Void Size and Frequency: The mix of underdense regions (blue) suggests matter inhomogeneities, implying moderate matter density.
3. Filament Thickness and Sharpness: N/A.
4. Fine-Scale Clumpiness: The distribution and intensity of colors, including blue, red, and yellow regions, imply moderate matter density and fluctuation

In [12]:
di = 2
print('===== CLAIMS =====')
for claim in new_results[di].claims:
    print(claim)

print('===== EXPERT CLAIMS =====')
for claim in new_results[di].expert_claims:
    print(claim)

===== CLAIMS =====
The map displays a mix of colors.
Gray indicates average density in the map.
Red suggests higher matter densities in the map.
Blue indicates lower densities in the map.
The presence of several yellow spots suggests significant fluctuations.
The yellow spots are above 2.9 standard deviations.
The mix of colors and notable structures like filament-like patterns imply a moderately heterogeneous matter distribution.
===== EXPERT CLAIMS =====
1. Lensing Peak (Cluster) Abundance: The presence of several yellow spots, which are above 2.9 standard deviations, suggests significant fluctuations.
2. Void Size and Frequency: The blue areas indicate lower densities, which could imply lower Omega_m allowing for these underdense regions.
3. Filament Thickness and Sharpness: Notable structures like filament-like patterns imply a moderately heterogeneous matter distribution.
4. Fine-Scale Clumpiness: The mix of colors and notable structures suggests a moderately heterogeneous matter 